# Lesson 31: Reinforcement Learning Basics

*ML Academy — companion notebook*

Markov decision processes, the Bellman equation, value iteration, policies, Q-learning with ε-greedy exploration, and reward design.

Run each cell in order with **Shift+Enter**. The explanations, maths and worked examples are in `lesson-31.html`; this notebook contains every code example from the lesson so you can run and modify it.

Requirements: `pip install numpy pandas scikit-learn matplotlib`

## 2. Markov decision processes

In [1]:
import numpy as np
ROWS, COLS = 4, 5
GOAL, PIT, WALL, START = (0, 4), (1, 4), (1, 1), (3, 0)
ACTIONS = {"↑": (-1, 0), "↓": (1, 0), "←": (0, -1), "→": (0, 1)}
STEP_COST, SLIP = -0.04, 0.1
states = [(i, j) for i in range(ROWS) for j in range(COLS) if (i, j) != WALL]
def move(s, a):
    i, j = s[0] + ACTIONS[a][0], s[1] + ACTIONS[a][1]
    return s if not (0 <= i < ROWS and 0 <= j < COLS) or (i, j) == WALL else (i, j)
def outcomes(s, a):                     # (probability, next state) — slips go sideways
    side = ["←", "→"] if a in "↑↓" else ["↑", "↓"]
    return [(1 - SLIP, move(s, a)), (SLIP / 2, move(s, side[0])), (SLIP / 2, move(s, side[1]))]
def reward(s):
    return 1.0 if s == GOAL else -1.0 if s == PIT else STEP_COST
terminal = {GOAL, PIT}
for i in range(ROWS):
    print(" ".join("G" if (i, j) == GOAL else "X" if (i, j) == PIT else "#" if (i, j) == WALL else "S" if (i, j) == START else "." for j in range(COLS)))

. . . . G
. # . . X
. . . . .
S . . . .


## 3. Values and the Bellman equation

In [2]:
def value_iteration(gamma=0.9, tol=1e-6):
    V = {s: 0.0 for s in states}
    for sweep in range(1, 1000):
        delta = 0
        for s in states:
            if s in terminal: continue
            best = max(sum(p * (reward(n) + (0 if n in terminal else gamma * V[n])) for p, n in outcomes(s, a)) for a in ACTIONS)
            delta, V[s] = max(delta, abs(best - V[s])), best
        if delta < tol:
            return V, sweep
V, sweeps = value_iteration()
print("converged after", sweeps, "sweeps")
for i in range(ROWS):
    print(" ".join(f"{V[(i, j)]:6.3f}" if (i, j) in V and (i, j) not in terminal else "   +1 " if (i, j) == GOAL else "   -1 " if (i, j) == PIT else "   ## " for j in range(COLS)))

converged after 13 sweeps
 0.558  0.682  0.815  0.973    +1 
 0.452    ##   0.684  0.731    -1 
 0.363  0.451  0.561  0.596  0.412
 0.283  0.363  0.453  0.480  0.385


## 4. From values to a policy

In [3]:
def greedy(V, gamma=0.9):
    return {s: max(ACTIONS, key=lambda a: sum(p * (reward(n) + (0 if n in terminal else gamma * V[n])) for p, n in outcomes(s, a)))
            for s in states if s not in terminal}
policy = greedy(V)
for i in range(ROWS):
    print(" ".join(policy.get((i, j), "G" if (i, j) == GOAL else "X" if (i, j) == PIT else "#") for j in range(COLS)))

→ → → → G
↑ # ↑ ↑ X
↑ → ↑ ↑ ←
→ → ↑ ↑ ←


## 6. Learning without a model: Q-learning

In [4]:
rng = np.random.default_rng(0)
def step(s, a):
    r, acc = rng.random(), 0
    for p, n in outcomes(s, a):
        acc += p
        if r < acc: return n, reward(n)
    return n, reward(n)
def q_learning(episodes=3000, alpha=0.1, gamma=0.9, eps=0.1):
    Q = {s: {a: 0.0 for a in ACTIONS} for s in states}
    for _ in range(episodes):
        s = START
        for t in range(100):
            a = rng.choice(list(ACTIONS)) if rng.random() < eps else max(Q[s], key=Q[s].get)
            n, r = step(s, a)
            target = r if n in terminal else r + gamma * max(Q[n].values())
            Q[s][a] += alpha * (target - Q[s][a])       # move the estimate towards the target
            s = n
            if s in terminal: break
    return Q
Q = q_learning()
learned = {s: max(Q[s], key=Q[s].get) for s in policy}
agree = np.mean([learned[s] == policy[s] for s in policy])
print("Q-learning's greedy policy matches value iteration in", f"{agree:.0%}", "of states")
print("estimated value of the start: Q-learning", round(max(Q[START].values()), 3), " value iteration", round(V[START], 3))

Q-learning's greedy policy matches value iteration in 94% of states
estimated value of the start: Q-learning 0.277  value iteration 0.283


## 7. Rewards shape behaviour

In [5]:
for cost in [-0.01, -0.4, -2.0]:
    STEP_COST = cost                                        # reward() reads this global
    pol = greedy(value_iteration()[0])
    rows = [" ".join(pol.get((i, j), "G" if (i, j) == GOAL else "X" if (i, j) == PIT else "#") for j in range(COLS)) for i in range(ROWS)]
    print(f"step cost {cost}:   " + "   |   ".join(rows))
STEP_COST = -0.04

step cost -0.01:   → → → → G   |   ↑ # ↑ ↑ X   |   ↑ → ↑ ↑ ←   |   ↑ → ↑ ↑ ←
step cost -0.4:   → → → → G   |   ↑ # ↑ ↑ X   |   ↑ → ↑ ↑ ←   |   → → ↑ ↑ ↑
step cost -2.0:   → → → → G   |   ↑ # → → X   |   → → → ↑ ↑   |   → → → ↑ ↑


## 9. In Python: Q-learning on a corridor

In [6]:
import numpy as np

rng = np.random.default_rng(1)
N, START, gamma, alpha, eps = 6, 2, 0.9, 0.2, 0.2
Q = np.zeros((N, 2))                      # actions: 0 left, 1 right
for episode in range(500):
    s = START
    while 0 < s < N - 1:
        a = rng.integers(2) if rng.random() < eps else int(Q[s].argmax())
        s2 = s + (1 if a == 1 else -1)
        r = 1.0 if s2 == N - 1 else -1.0 if s2 == 0 else 0.0
        target = r if s2 in (0, N - 1) else r + gamma * Q[s2].max()
        Q[s, a] += alpha * (target - Q[s, a])
        s = s2
print("Q values for 'right' in squares 1 to 4:", np.round(Q[1:N - 1, 1], 3).tolist())
print("best action in squares 1 to 4:", ["right" if q[1] > q[0] else "left" for q in Q[1:N - 1]])

Q values for 'right' in squares 1 to 4: [0.729, 0.81, 0.9, 1.0]
best action in squares 1 to 4: ['right', 'right', 'right', 'right']


## 12. Going deeper: SARSA versus Q-learning on a cliff

In [7]:
import numpy as np
# Cliff walking: a 4×8 grid; the bottom row between start and goal is a cliff (−100, back to start).
R, C = 4, 8
start, goal = (3, 0), (3, 7)
moves = [(-1, 0), (1, 0), (0, -1), (0, 1)]
def env(s, a):
    i, j = min(max(s[0] + moves[a][0], 0), R - 1), min(max(s[1] + moves[a][1], 0), C - 1)
    if i == 3 and 0 < j < 7: return start, -100
    return (i, j), -1
rng = np.random.default_rng(0)
def train(method, episodes=500, alpha=0.5, eps=0.1):
    Q = np.zeros((R, C, 4))
    pick = lambda s: rng.integers(4) if rng.random() < eps else int(Q[s].argmax())
    for _ in range(episodes):
        s = start; a = pick(s)
        for t in range(200):
            s2, r = env(s, a); a2 = pick(s2)
            nxt = 0 if s2 == goal else (Q[s2].max() if method == "Q" else Q[s2][a2])
            Q[s][a] += alpha * (r + nxt - Q[s][a]); s, a = s2, a2
            if s == goal: break
    return Q
def route(Q):
    s, path = start, [start]
    for _ in range(40):
        s, _ = env(s, int(Q[s].argmax())); path.append(s)
        if s == goal: break
    return path
for method in ["Q", "SARSA"]:
    p = route(train(method))
    edge = sum(1 for c in p if c[0] == 2)                  # steps right next to the cliff
    print(f"{method:5s}: greedy route length {len(p) - 1}, steps along the cliff edge {edge}")

Q    : greedy route length 9, steps along the cliff edge 8
SARSA: greedy route length 13, steps along the cliff edge 2


In [8]:
def run_returns(method, episodes=500, eps=0.1):
    Q, total = np.zeros((R, C, 4)), []
    for _ in range(episodes):
        s, G = start, 0
        a = rng.integers(4) if rng.random() < eps else int(Q[s].argmax())
        for t in range(200):
            s2, r = env(s, a); G += r
            a2 = rng.integers(4) if rng.random() < eps else int(Q[s2].argmax())
            nxt = 0 if s2 == goal else (Q[s2].max() if method == "Q" else Q[s2][a2])
            Q[s][a] += 0.5 * (r + nxt - Q[s][a]); s, a = s2, a2
            if s == goal: break
        total.append(G)
    return np.mean(total[-200:])
q_ret = np.mean([run_returns("Q") for _ in range(5)]); s_ret = np.mean([run_returns("SARSA") for _ in range(5)])
print("average return while still exploring: Q-learning", round(q_ret, 1), "  SARSA", round(s_ret, 1))

average return while still exploring: Q-learning -31.2   SARSA -21.0


## Running example: Chennai Homes

The same 12-house dataset used in every lesson (`data/houses.csv` in the course folder).

In [9]:
import pandas as pd
import numpy as np
df = pd.read_csv("../data/houses.csv")
# Each week a listed home sells with probability p; cutting the price 5% raises p but lowers the sale value.
p_hold = df.sold_fast.mean() / 4                     # about half the homes sell within ~4 weeks
p_cut, price, gamma, weekly_cost = min(1, 2 * p_hold), df.price_lakh.median(), 0.98, 0.5
V = {k: 0.0 for k in range(4)}                       # state: number of price cuts so far (max 3)
for _ in range(500):
    V = {k: max(p_hold * price * 0.95 ** k + (1 - p_hold) * gamma * V[k] - weekly_cost,
                p_cut * price * 0.95 ** (k + 1) + (1 - p_cut) * gamma * V[min(k + 1, 3)] - weekly_cost if k < 3 else -1e9)
         for k in V}
for k in V:
    hold = p_hold * price * 0.95 ** k + (1 - p_hold) * gamma * V[k] - weekly_cost
    print(f"after {k} cuts: value {V[k]:.1f} lakh, best action:", "hold" if abs(V[k] - hold) < 1e-9 else "cut")

after 0 cuts: value 62.7 lakh, best action: hold
after 1 cuts: value 59.4 lakh, best action: hold
after 2 cuts: value 56.3 lakh, best action: hold
after 3 cuts: value 53.3 lakh, best action: hold


## Coding challenges

Each cell below has **one missing line**, marked `# ← your code here` (or `# ← replace`). Write the line, run the cell, and the check at the bottom tells you whether you got it right. Stuck? A wrong answer prints a hint, and the solutions are at the very end of this notebook.

In [10]:
# Checker used by the challenges below — run this cell first.
import numpy as np
def check(got, want, hint='', atol=1e-6):
    try:
        ok = got is not None and np.shape(got) == np.shape(want) and np.allclose(np.asarray(got, dtype=float), np.asarray(want, dtype=float), rtol=1e-4, atol=atol)
    except Exception:
        ok = False
    print('✅ Correct!' if ok else '❌ Not yet. Hint: ' + hint)


### Challenge 1: A discounted return

Compute the discounted return of the rewards with γ = 0.5.

In [11]:
rewards = [2, 4, 8]
gamma = 0.5
ret = sum(rewards)  # ← discount each reward
check(ret, 6.0, hint='Multiply each reward by γ to the power of its step.')

❌ Not yet. Hint: Multiply each reward by γ to the power of its step.


### Challenge 2: A Q-learning update

Apply one Q-learning update.

In [12]:
q, r, best_next, alpha, gamma = 1.0, 0.5, 2.0, 0.5, 0.9
q_new = r + gamma * best_next  # ← move only a fraction alpha of the way
check(q_new, 1.65, hint='Move q towards r + γ × best_next by a fraction α.')

❌ Not yet. Hint: Move q towards r + γ × best_next by a fraction α.


## Your turn

Try the **practice exercises** at the end of the lesson page. Many of them ask you to change one of the cells above and re-run it.

## Challenge solutions

*Spoilers — try the challenges first!*

**1. A discounted return** — the missing line:

```python
ret = sum(r * gamma ** t for t, r in enumerate(rewards))
```

**2. A Q-learning update** — the missing line:

```python
q_new = q + alpha * (r + gamma * best_next - q)
```